In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from delta.tables import DeltaTable

# Read existing silver table
df_silver = spark.read.table("fraud_analytics.silver.credit_card_fraud")

# Simulate batch: take 500 existing rows + modify some values
df_existing_modified = df_silver.limit(500) \
    .withColumn("amount", col("amount") * 1.10) \
    .withColumn("transaction_type", lit("purchase"))

# Simulate 10 completely new transactions
df_new = df_silver.limit(10) \
    .withColumn("transaction_id", col("transaction_id") + 999990) \
    .withColumn("amount", lit(9999.99)) \
    .withColumn("is_fraud", lit(True))

# Combine into one new batch
df_new_batch = df_existing_modified.union(df_new)

print(f"New batch row count: {df_new_batch.count()}")
print(f"Modified existing: 500 rows")
print(f"Brand new transactions: 10 rows")

In [0]:
# Get reference to existing Silver Delta table
silver_table = DeltaTable.forName(spark, "fraud_analytics.silver.credit_card_fraud")

# MERGE new batch into Silver
silver_table.alias("target") \
    .merge(
        df_new_batch.alias("source"),
        "target.transaction_id = source.transaction_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("MERGE completed successfully!")

In [0]:
# Check new row count — should be 100,010 (100K original + 10 new)
df_after = spark.read.table("fraud_analytics.silver.credit_card_fraud")
print(f"Row count after MERGE: {df_after.count()}")

# Verify new transactions exist
spark.sql("""
    SELECT transaction_id, amount, is_fraud
    FROM fraud_analytics.silver.credit_card_fraud
    WHERE transaction_id > 999990
""").show()

# Verify updates applied — amount should be 10% higher for modified rows
spark.sql("""
    SELECT transaction_id, amount
    FROM fraud_analytics.silver.credit_card_fraud
    WHERE transaction_id <= 500
    LIMIT 5
""").show()

In [0]:
# See new version created by MERGE
spark.sql("""
    DESCRIBE HISTORY fraud_analytics.silver.credit_card_fraud
""").select("version", "timestamp", "operation", "operationMetrics").display()